Calculate heats of formation from a basis.

In [3]:
import pint
import re
import numpy as np

term_pattern = re.compile(r"([A-Z][a-z]?)(\d*)")

elements = ("C", "H", "O")
basis = ("H2", "CH4", "H2O")

# All units in Hartree (converted from kcal/mol)
basis_formation_enthalpy = {
    "H2": 0.0 * 0.0015936,
    "CH4": -15.906 * 0.0015936,
    "H2O": -57.106 * 0.0015936,
}


def stoichiometry(formula: str) -> dict[str, int]:
    """Parse stoichiometry from formula."""
    matches = term_pattern.findall(formula)
    stoichiometry = {}
    for element, count in matches:
        count = int(count) if count else 1
        stoichiometry[element] = count
    return stoichiometry


def stoichiometry_vector(formula: str) -> list[int]:
    """Get stoichiometry vector for a given formula."""
    stoich = stoichiometry(formula)
    return [stoich.get(element, 0) for element in elements]

basis_matrix = np.column_stack(tuple(stoichiometry_vector(formula) for formula in basis))


def basis_coefficients(formula: str) -> dicst[str, int]:
    """Calculate basis coefficients for a given formula."""
    coeffs =  np.linalg.solve(basis_matrix, stoichiometry_vector(formula))
    return {k: c for k, c in zip(basis, coeffs.tolist(), strict=True)}


def formation_enthalpy(formula: str, absolute_enthalpy: float, basis_absolute_enthalpy: dict[str, float]) -> float:
    """Calculate the shift from electronic energy to heat of formation."""
    coeff = basis_coefficients(formula)
    reference_correction = sum(coeff[k] * (basis_formation_enthalpy[k] - basis_absolute_enthalpy[k]) for k in basis)
    return absolute_enthalpy + reference_correction

basis_absolute_enthalpy = {
    "H2": -1.163339464,
    "CH4": -40.40341344,
    "H2O": -76.33468352,
}

formation_enthalpy("C5H8", -194.8901193, basis_absolute_enthalpy=basis_absolute_enthalpy)

0.020172108000025446